<a href="https://colab.research.google.com/github/sokrypton/ml4me/blob/main/finetune_esm2_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Fine-tuning ESM2 for Protein Binding Prediction**

A lesson in three acts:

| Strategy | What's trainable | Parameters | Risk |
|----------|-----------------|------------|------|
| **Step 1:** Head only | Classification head | ~hundreds | Underfitting |
| **Step 2:** Full fine-tune | Everything | ~8M | Overfitting |
| **Step 3:** LoRA | Adapters + head | ~thousands | Sweet spot |

We'll train all three on the same data and compare loss curves to see the tradeoffs.

Special thanks to [Amelie Schreiber](https://github.com/Amelie-Schreiber/esm2_loras) and [Sergey Ovchinnikov](https://github.com/sokrypton/roscon2024)

---
## Setup

In [10]:
!pip install -q transformers peft accelerate
!pip install -U torchao>=0.16.0

In [2]:
model_name = "esm2_t6_8M_UR50D" # @param ["esm2_t33_650M_UR50D", "esm2_t30_150M_UR50D", "esm2_t12_35M_UR50D", "esm2_t6_8M_UR50D"]

In [3]:
import os
import copy
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import EsmForTokenClassification, AutoTokenizer
from peft import LoraConfig, get_peft_model

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cuda:0


---
## Load Data

We use the AF2Bind dataset — per-residue binding labels for proteins.

In [4]:
#@markdown ##Data Loading
batch_size = 32 # @param {"type":"integer"}
max_crop_len = 512 # @param {"type":"integer"}

!wget -qnc https://github.com/sokrypton/roscon2024/raw/main/af2bind_data_0.pkl
import pickle
with open("af2bind_data_0.pkl", "rb") as handle:
  DATA = pickle.load(handle)

def pad_sequence(seq, max_len, pad_value=0):
    pad_size = max(0, max_len - len(seq))
    return np.pad(seq, (0, pad_size), 'constant', constant_values=pad_value)[:max_len]

class CustomProteinDataset(Dataset):
    def __init__(self, inputs, attention_masks, outputs, masks, max_crop_len=128):
        self.inputs = inputs
        self.attention_masks = attention_masks
        self.outputs = outputs
        self.masks = masks
        self.max_crop_len = max_crop_len

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        input_ids = self.inputs[idx]
        attention_mask = self.attention_masks[idx]
        output = self.outputs[idx]
        mask = self.masks[idx]

        true_len = int(np.sum(attention_mask))
        crop_len = min(self.max_crop_len, true_len)

        if true_len > crop_len:
            start_idx = np.random.randint(0, true_len - crop_len + 1)
        else:
            start_idx = 0

        input_ids = input_ids[start_idx:start_idx + crop_len]
        attention_mask = attention_mask[start_idx:start_idx + crop_len].astype(np.float32)
        output = output[start_idx:start_idx + crop_len].astype(np.float32)
        mask = mask[start_idx:start_idx + crop_len].astype(np.float32)

        input_ids = pad_sequence(input_ids, self.max_crop_len)
        attention_mask = pad_sequence(attention_mask, self.max_crop_len)
        output = pad_sequence(output, self.max_crop_len)
        mask = pad_sequence(mask, self.max_crop_len)

        return torch.tensor(input_ids), torch.tensor(attention_mask), torch.tensor(output), torch.tensor(mask)

dataloaders = []
for v in range(3):  # train/test/validation
    dataset = CustomProteinDataset(DATA["inputs"][v], DATA["attention_masks"][v],
                                   DATA["outputs"][v], DATA["masks"][v],
                                   max_crop_len=max_crop_len)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=(v == 0))
    dataloaders.append(dataloader)

train_loader, test_loader, val_loader = dataloaders
print(f"Train: {len(train_loader.dataset)} | Test: {len(test_loader.dataset)} | Val: {len(val_loader.dataset)}")

Train: 582 | Test: 23 | Val: 67


---
## Training Utilities

In [5]:
def compute_loss(logits, labels, mask):
    """Compute masked binary cross-entropy loss."""
    loss = nn.BCEWithLogitsLoss(reduction='none')(logits, labels)
    masked_loss = loss * mask
    return masked_loss.sum() / mask.sum()

def train_one_epoch(model, dataloader, optimizer):
    """Train for one epoch, return average loss."""
    model.train()
    total_loss = 0
    for batch in dataloader:
        inputs, attn_mask, labels, mask = [x.to(DEVICE) for x in batch]
        logits = model(input_ids=inputs, attention_mask=attn_mask).logits.squeeze(-1)
        loss = compute_loss(logits, labels, mask)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

@torch.no_grad()
def validate(model, dataloader):
    """Validate, return average loss."""
    model.eval()
    total_loss = 0
    for batch in dataloader:
        inputs, attn_mask, labels, mask = [x.to(DEVICE) for x in batch]
        logits = model(input_ids=inputs, attention_mask=attn_mask).logits.squeeze(-1)
        total_loss += compute_loss(logits, labels, mask).item()
    return total_loss / len(dataloader)

def train_and_record(model, train_loader, test_loader, num_epochs, lr, label):
    """Train model, return dict of train/test loss curves."""
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    train_losses, test_losses = [], []

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    print(f"{'='*60}")

    for epoch in range(num_epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer)
        test_loss = validate(model, test_loader)
        train_losses.append(train_loss)
        test_losses.append(test_loss)
        print(f"  Epoch {epoch+1:2d}/{num_epochs} | Train: {train_loss:.4f} | Test: {test_loss:.4f}")

    return {"train": train_losses, "test": test_losses}

In [6]:
# Training config
num_epochs = 20  # @param {"type":"integer"}
learning_rate = 1e-3  # @param {"type":"number"}

# We'll collect all results here for the final comparison plot
all_results = {}

---
## Step 1: Head Only (Frozen Backbone)

Freeze the entire ESM2 transformer and only train the classification head.
This is fast and cheap, but the model can only learn a linear mapping from
pretrained features — it may **underfit** if those features aren't rich enough
for the task.

In [7]:
# Load fresh model
model_head = EsmForTokenClassification.from_pretrained(
    f"facebook/{model_name}", num_labels=1, hidden_dropout_prob=0.15
).to(DEVICE)

# Freeze backbone — only classifier head remains trainable
for param in model_head.esm.parameters():
    param.requires_grad = False

all_results["Head Only"] = train_and_record(
    model_head, train_loader, test_loader, num_epochs, learning_rate,
    "Step 1: Head Only (frozen backbone)"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmForTokenClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Step 1: Head Only (frozen backbone)
Trainable: 321 / 7,409,402 (0.00%)
  Epoch  1/20 | Train: 0.5700 | Test: 0.4914
  Epoch  2/20 | Train: 0.4433 | Test: 0.4000
  Epoch  3/20 | Train: 0.3827 | Test: 0.3531
  Epoch  4/20 | Train: 0.3554 | Test: 0.3296
  Epoch  5/20 | Train: 0.3431 | Test: 0.3169
  Epoch  6/20 | Train: 0.3416 | Test: 0.3092
  Epoch  7/20 | Train: 0.3328 | Test: 0.3048
  Epoch  8/20 | Train: 0.3334 | Test: 0.3016
  Epoch  9/20 | Train: 0.3290 | Test: 0.2992
  Epoch 10/20 | Train: 0.3284 | Test: 0.2974
  Epoch 11/20 | Train: 0.3294 | Test: 0.2957
  Epoch 12/20 | Train: 0.3283 | Test: 0.2946
  Epoch 13/20 | Train: 0.3298 | Test: 0.2936
  Epoch 14/20 | Train: 0.3262 | Test: 0.2927
  Epoch 15/20 | Train: 0.3269 | Test: 0.2916
  Epoch 16/20 | Train: 0.3230 | Test: 0.2905
  Epoch 17/20 | Train: 0.3246 | Test: 0.2894
  Epoch 18/20 | Train: 0.3268 | Test: 0.2885
  Epoch 19/20 | Train: 0.3220 | Test: 0.2881
  Epoch 20/20 | Train: 0.3214 | Test: 0.2871


---
## Step 2: Full Fine-tune

Unfreeze everything — the backbone adapts to the task. More expressive, but
with ~8M trainable parameters on a small dataset, watch the gap between
train and test loss grow. That gap is **overfitting**.

In [8]:
# Load fresh model — everything trainable by default
model_full = EsmForTokenClassification.from_pretrained(
    f"facebook/{model_name}", num_labels=1, hidden_dropout_prob=0.15
).to(DEVICE)

all_results["Full Fine-tune"] = train_and_record(
    model_full, train_loader, test_loader, num_epochs, learning_rate,
    "Step 2: Full Fine-tune (all parameters)"
)

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmForTokenClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Step 2: Full Fine-tune (all parameters)
Trainable: 7,409,402 / 7,409,402 (100.00%)
  Epoch  1/20 | Train: 0.3483 | Test: 0.2881
  Epoch  2/20 | Train: 0.3178 | Test: 0.2852
  Epoch  3/20 | Train: 0.3169 | Test: 0.2822
  Epoch  4/20 | Train: 0.3100 | Test: 0.2742
  Epoch  5/20 | Train: 0.2987 | Test: 0.2706
  Epoch  6/20 | Train: 0.2948 | Test: 0.2763
  Epoch  7/20 | Train: 0.2887 | Test: 0.2663
  Epoch  8/20 | Train: 0.2809 | Test: 0.2696
  Epoch  9/20 | Train: 0.2811 | Test: 0.2741
  Epoch 10/20 | Train: 0.2675 | Test: 0.2803
  Epoch 11/20 | Train: 0.2616 | Test: 0.2838
  Epoch 12/20 | Train: 0.2502 | Test: 0.2869
  Epoch 13/20 | Train: 0.2392 | Test: 0.3065
  Epoch 14/20 | Train: 0.2236 | Test: 0.3081
  Epoch 15/20 | Train: 0.2171 | Test: 0.3201
  Epoch 16/20 | Train: 0.2008 | Test: 0.3252
  Epoch 17/20 | Train: 0.1862 | Test: 0.3433
  Epoch 18/20 | Train: 0.1811 | Test: 0.3213
  Epoch 19/20 | Train: 0.1713 | Test: 0.3572
  Epoch 20/20 | Train: 0.1609 | Test: 0.3598


---
## Step 3: LoRA

Low-Rank Adaptation: inject small trainable matrices into the attention layers.
The backbone stays frozen, but the model can now learn task-specific attention
patterns with far fewer parameters. This is the **sweet spot** — more expressive
than head-only, but regularized enough to avoid overfitting.

In [11]:
# Load fresh model
model_lora = EsmForTokenClassification.from_pretrained(
    f"facebook/{model_name}", num_labels=1, hidden_dropout_prob=0.15
).to(DEVICE)

# Apply LoRA
lora_config = LoraConfig(
    r=4,
    lora_alpha=16,
    target_modules=["query", "key", "value"],
    lora_dropout=0.15,
    bias="none",
    task_type="TOKEN_CLS",
    modules_to_save=["classifier"],  # also save the head with the adapter
)
model_lora = get_peft_model(model_lora, lora_config)

all_results["LoRA"] = train_and_record(
    model_lora, train_loader, test_loader, num_epochs, learning_rate,
    "Step 3: LoRA (adapters + head)"
)

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmForTokenClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Step 3: LoRA (adapters + head)
Trainable: 46,401 / 7,455,803 (0.62%)
  Epoch  1/20 | Train: 0.4952 | Test: 0.2852
  Epoch  2/20 | Train: 0.3206 | Test: 0.2865
  Epoch  3/20 | Train: 0.3192 | Test: 0.2855
  Epoch  4/20 | Train: 0.3192 | Test: 0.2856
  Epoch  5/20 | Train: 0.3182 | Test: 0.2849
  Epoch  6/20 | Train: 0.3191 | Test: 0.2849
  Epoch  7/20 | Train: 0.3169 | Test: 0.2837
  Epoch  8/20 | Train: 0.3195 | Test: 0.2820
  Epoch  9/20 | Train: 0.3155 | Test: 0.2795
  Epoch 10/20 | Train: 0.3129 | Test: 0.2785
  Epoch 11/20 | Train: 0.3039 | Test: 0.2756
  Epoch 12/20 | Train: 0.3018 | Test: 0.2767
  Epoch 13/20 | Train: 0.3000 | Test: 0.2736
  Epoch 14/20 | Train: 0.3015 | Test: 0.2738
  Epoch 15/20 | Train: 0.2954 | Test: 0.2751
  Epoch 16/20 | Train: 0.2982 | Test: 0.2717
  Epoch 17/20 | Train: 0.2934 | Test: 0.2693
  Epoch 18/20 | Train: 0.2957 | Test: 0.2737
  Epoch 19/20 | Train: 0.2909 | Test: 0.2738
  Epoch 20/20 | Train: 0.2957 | Test: 0.2664


---
## Comparison

Now let's plot all three strategies side by side.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = {"Head Only": "#4CAF50", "Full Fine-tune": "#F44336", "LoRA": "#2196F3"}
epochs = range(1, num_epochs + 1)

# Individual plots
for ax, (name, res) in zip(axes, all_results.items()):
    ax.plot(epochs, res["train"], '-', color=colors[name], label="Train", linewidth=2)
    ax.plot(epochs, res["test"], '--', color=colors[name], label="Test", linewidth=2)
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Annotate the gap
    gap = res["test"][-1] - res["train"][-1]
    ax.annotate(f"gap: {gap:.4f}",
                xy=(num_epochs, res["test"][-1]),
                fontsize=10, ha='right', va='bottom',
                color=colors[name])

plt.suptitle(f"Fine-tuning Strategies for {model_name}", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Overlay: test loss comparison
plt.figure(figsize=(8, 5))
for name, res in all_results.items():
    plt.plot(epochs, res["test"], '-o', color=colors[name], label=name, linewidth=2, markersize=4)

plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Test Loss", fontsize=12)
plt.title("Test Loss Comparison", fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Summary table
print(f"{'Strategy':<20} {'Train Loss':>12} {'Test Loss':>12} {'Gap':>10} {'Trainable':>12}")
print("-" * 70)
for name, res in all_results.items():
    train_final = res["train"][-1]
    test_final = res["test"][-1]
    gap = test_final - train_final
    print(f"{name:<20} {train_final:>12.4f} {test_final:>12.4f} {gap:>10.4f}")

---
## What to look for

- **Head Only**: Train and test loss converge together but plateau high → the model
  doesn't have enough capacity to learn the task well. This is **underfitting**.

- **Full Fine-tune**: Train loss drops very low, but test loss starts rising after
  a few epochs → the model memorizes training data instead of generalizing.
  The growing train/test **gap** is the signature of **overfitting**.

- **LoRA**: Train loss drops reasonably, test loss tracks it closely → the low-rank
  constraint acts as implicit regularization, giving us the best of both worlds.

### Key takeaway
LoRA lets the model adapt its attention patterns to the task (unlike head-only)
without the freedom to memorize noise (unlike full fine-tuning). For small
biological datasets, this is usually the right call.